# HRFLM Evaluation -- Global Explainability Metrics

Evaluates the HRFLM (Hybrid Random Forest + Linear Model) global surrogate built in `HRFLM.ipynb`, using the SAME SIX metrics as `lime_eval.ipynb`, but computed GLOBALLY across the entire reference population instead of for one patient at a time -- consistent with HRFLM's role as a **global** explanation method (audience: admin only).

| Metric | LIME (local, per-patient) | HRFLM (global, population-wide) |
|---|---|---|
| Fidelity | R^2 of the local linear surrogate around ONE patient | R^2 of the hybrid RF+LM surrogate's probability vs. the real model's probability, across ALL reference patients |
| Accuracy Gain | Surrogate accuracy vs. baseline, on a small sampled set | Surrogate accuracy vs. baseline, across the FULL reference dataset |
| Agreement | Surrogate vs. real model class match, for one patient at a time (averaged) | Surrogate vs. real model class match, across ALL reference patients at once |
| Stability | Repeated LIME runs on the SAME patient | Repeated HRFLM re-fits (different random seeds/bootstrap draws) on the SAME data, comparing global feature-importance vectors |
| Sparsity | Gini coefficient of one patient's local contribution weights | Gini coefficient of the single global feature-importance vector |
| Deletion / Insertion AUC | Progressively baseline/restore features per-patient, ranked by that patient's LIME weights, tracking the real model's own confidence | Progressively baseline/restore features across ALL reference patients at once, ranked by the single global hybrid_importance vector |
| Reward Improvement | Real PPO reward vs. LIME surrogate reward, on a sample of patients | Real PPO reward vs. HRFLM surrogate reward, across the full reference dataset |

### 1. Import Libraries

In [ ]:
import os
import json

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

import warnings
warnings.filterwarnings("ignore")

### 2. Device

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running Device :", DEVICE)

### 3. Canonical Feature Contract

In [ ]:
CANONICAL_FEATURE_ORDER = [
    "thalach", "restecg", "oldpeak", "slope", "age",
    "sex", "cp", "exang", "trestbps", "fbs"
]

FEATURE_LABELS = {
    "thalach": "Maximum heart rate",
    "restecg": "Resting ECG result",
    "oldpeak": "ECG stress-test change",
    "slope": "ECG ST-segment slope",
    "age": "Age",
    "sex": "Sex",
    "cp": "Chest pain type",
    "exang": "Exercise-related chest discomfort",
    "trestbps": "Resting blood pressure",
    "fbs": "Fasting blood sugar indicator",
}

### 4. Preprocessing Parameters (copied from `models/rl/preprocessing_params.json`)

Identical to `HRFLM.ipynb` and `lime_eval.ipynb` -- see those notebooks for the byte-exact verification against the original training data.

In [ ]:
ROBUST_SCALER_PARAMS = {
    "age":      {"median": 56.0,  "iqr": 12.0},
    "trestbps": {"median": 130.0, "iqr": 22.0},
    "thalach":  {"median": 140.0, "iqr": 38.5},
    "oldpeak":  {"median": 1.0,   "iqr": 1.9},
}

CP_INT_TO_ONEHOT = {
    0: "cp_typical angina",
    1: "cp_atypical angina",
    2: "cp_non-anginal",
    3: "cp_asymptomatic",
}
RESTECG_INT_TO_ONEHOT = {
    0: "restecg_normal",
    1: "restecg_st-t abnormality",
    2: "restecg_lv hypertrophy",
}

TRAINED_FEATURE_ORDER_15DIM = [
    "sex", "fbs", "exang", "age", "trestbps", "thalach", "oldpeak",
    "cp_asymptomatic", "cp_atypical angina", "cp_non-anginal", "cp_typical angina",
    "restecg_lv hypertrophy", "restecg_normal", "restecg_st-t abnormality",
    "slope",
]


def to_trained_representation(features: dict) -> np.ndarray:
    """Raw 10 canonical features -> the 15-dim vector the RL models were
    trained on. Direct port of backend/services/preprocessingService.js."""
    vector = {}
    vector["sex"] = features["sex"]
    vector["fbs"] = features["fbs"]
    vector["exang"] = features["exang"]
    vector["slope"] = features["slope"]

    for key in ("age", "trestbps", "thalach", "oldpeak"):
        params = ROBUST_SCALER_PARAMS[key]
        vector[key] = (features[key] - params["median"]) / params["iqr"]

    for column in ["cp_asymptomatic", "cp_atypical angina", "cp_non-anginal", "cp_typical angina",
                   "restecg_lv hypertrophy", "restecg_normal", "restecg_st-t abnormality"]:
        vector[column] = 0
    vector[CP_INT_TO_ONEHOT[features["cp"]]] = 1
    vector[RESTECG_INT_TO_ONEHOT[features["restecg"]]] = 1

    return np.array([vector[key] for key in TRAINED_FEATURE_ORDER_15DIM], dtype=np.float32)

### 5. Load the Trained PPO Model

In [ ]:
HIDDEN_DIM = 128
ACTION_SIZE = 2
STATE_SIZE = 15

PPO_MODEL_PATH = "../../trained_models/ppo.pth"

if not os.path.exists(PPO_MODEL_PATH):
    raise FileNotFoundError(f"{PPO_MODEL_PATH} not found. Update PPO_MODEL_PATH.")


class ActorCritic(nn.Module):
    def __init__(self, state_size):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_size, HIDDEN_DIM), nn.ReLU(),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.ReLU(),
        )
        self.actor = nn.Sequential(
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.ReLU(),
            nn.Linear(HIDDEN_DIM, ACTION_SIZE), nn.Softmax(dim=-1),
        )
        self.critic = nn.Sequential(
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.ReLU(),
            nn.Linear(HIDDEN_DIM, 1),
        )
        self.to(DEVICE)

    def forward(self, state):
        if not torch.is_tensor(state):
            state = torch.FloatTensor(state)
        state = state.to(DEVICE)
        features = self.shared(state)
        return self.actor(features), self.critic(features)


ppo_model = ActorCritic(STATE_SIZE)
checkpoint = torch.load(PPO_MODEL_PATH, map_location=DEVICE)
ppo_model.load_state_dict(checkpoint["actor_state_dict"])
ppo_model.critic.load_state_dict(checkpoint["critic_state_dict"])
ppo_model.eval()

print("PPO model loaded from:", PPO_MODEL_PATH)

### 6. Prediction Function

In [ ]:
def predict_probability_from_raw(raw_feature_rows: np.ndarray) -> np.ndarray:
    """raw_feature_rows: shape (n, 10), CANONICAL_FEATURE_ORDER, raw values."""
    preprocessed_rows = []
    for row in raw_feature_rows:
        features = dict(zip(CANONICAL_FEATURE_ORDER, row))
        for key in ("restecg", "slope", "sex", "cp", "exang", "fbs"):
            features[key] = int(round(features[key]))
        preprocessed_rows.append(to_trained_representation(features))

    batch = torch.FloatTensor(np.stack(preprocessed_rows)).to(DEVICE)
    with torch.no_grad():
        action_probabilities, _ = ppo_model(batch)
    return action_probabilities.cpu().numpy()

### 7. Load Reference Data and Real Model Predictions

In [ ]:
REFERENCE_DATA_PATH = "../data/heart_disease_trimmed.csv"

reference_df = pd.read_csv(REFERENCE_DATA_PATH)
X_raw = reference_df[CANONICAL_FEATURE_ORDER].to_numpy(dtype=np.float32)
y_true = reference_df["target"].to_numpy(dtype=np.int64)

real_model_probabilities = predict_probability_from_raw(X_raw)
real_model_predictions = np.argmax(real_model_probabilities, axis=1)

print("Reference data shape:", X_raw.shape)
print("Real model predicted-positive rate:", round(float(real_model_predictions.mean()), 4))

### 8. Train the HRFLM Hybrid Surrogate (baseline, seed=42)

Same training procedure as `HRFLM.ipynb` -- trained against the real model's own predictions.

In [ ]:
RANDOM_SEED = 42


def train_hrflm_surrogate(X, y_target, random_state=RANDOM_SEED, n_estimators=200, max_depth=6):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    random_forest = RandomForestClassifier(
        n_estimators=n_estimators, max_depth=max_depth, random_state=random_state, class_weight="balanced"
    )
    random_forest.fit(X, y_target)

    linear_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=random_state)
    linear_model.fit(X_scaled, y_target)

    return scaler, random_forest, linear_model


def hrflm_predict_probability(scaler, random_forest, linear_model, raw_rows):
    rf_probabilities = random_forest.predict_proba(raw_rows)
    lr_probabilities = linear_model.predict_proba(scaler.transform(raw_rows))
    return (rf_probabilities + lr_probabilities) / 2.0


def global_feature_importance_vector(random_forest, linear_model):
    rf_importance = random_forest.feature_importances_
    rf_importance = rf_importance / rf_importance.sum() if rf_importance.sum() > 0 else rf_importance

    lr_importance = np.abs(linear_model.coef_[0])
    lr_importance = lr_importance / lr_importance.sum() if lr_importance.sum() > 0 else lr_importance

    return (rf_importance + lr_importance) / 2.0


scaler, random_forest, linear_model = train_hrflm_surrogate(X_raw, real_model_predictions)

hybrid_probabilities = hrflm_predict_probability(scaler, random_forest, linear_model, X_raw)
hybrid_predictions = np.argmax(hybrid_probabilities, axis=1)
hybrid_importance = global_feature_importance_vector(random_forest, linear_model)

print("HRFLM surrogate trained (seed =", RANDOM_SEED, ")")
print("Hybrid probability shape:", hybrid_probabilities.shape)

### 9. Fidelity, Agreement, and Accuracy Gain (Global)

**Fidelity** here is the R^2 of the surrogate's predicted probability of the positive class against the real model's predicted probability of the positive class, across ALL reference rows -- the global analog of LIME's per-instance local R^2.

In [ ]:
# Fidelity: R^2 of surrogate probability vs. real model probability (class 1), globally.
global_fidelity = r2_score(real_model_probabilities[:, 1], hybrid_probabilities[:, 1])

# Agreement: fraction of ALL rows where surrogate predicted class == real model predicted class.
global_agreement = float(np.mean(hybrid_predictions == real_model_predictions))

# Accuracy Gain: surrogate accuracy vs. true labels, minus majority-class baseline, globally.
surrogate_accuracy = float(np.mean(hybrid_predictions == y_true))
majority_class = int(np.round(np.mean(y_true)))
baseline_accuracy = float(np.mean(np.full_like(y_true, majority_class) == y_true))
global_accuracy_gain = surrogate_accuracy - baseline_accuracy

print("=" * 60)
print("FIDELITY, AGREEMENT, ACCURACY GAIN (GLOBAL)")
print("=" * 60)
print(f"Global Fidelity (R^2)                : {global_fidelity:.4f}")
print(f"Global Agreement                     : {global_agreement:.4f}")
print(f"Surrogate Accuracy (vs. true labels) : {surrogate_accuracy:.4f}")
print(f"Majority-Class Baseline Accuracy     : {baseline_accuracy:.4f}")
print(f"Accuracy Gain (surrogate - baseline) : {global_accuracy_gain:+.4f}")
print("=" * 60)

### 10. Stability (Global)

Re-trains the HRFLM surrogate multiple times with different random seeds (different Random Forest bootstrap draws and Logistic Regression initialization), each time on the SAME data and SAME surrogate target (the real model's predictions), then measures the cosine similarity of the resulting GLOBAL feature-importance vectors across refits. High stability means the global explanation does not depend on incidental training randomness.

In [ ]:
NUM_STABILITY_REFITS = 5
STABILITY_SEEDS = [1, 2, 3, 4, 5]


def cosine_similarity(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


refit_importance_vectors = []
for seed in STABILITY_SEEDS:
    _, rf_refit, lr_refit = train_hrflm_surrogate(X_raw, real_model_predictions, random_state=seed)
    refit_importance_vectors.append(global_feature_importance_vector(rf_refit, lr_refit))

pairwise_similarities = [
    cosine_similarity(refit_importance_vectors[i], refit_importance_vectors[j])
    for i in range(len(refit_importance_vectors))
    for j in range(i + 1, len(refit_importance_vectors))
]
global_stability = float(np.mean(pairwise_similarities))

print("=" * 60)
print("STABILITY (GLOBAL)")
print("=" * 60)
print(f"Refits compared           : {NUM_STABILITY_REFITS}")
print(f"Mean Stability (cosine)   : {global_stability:.4f}")
print("=" * 60)

### 11. Sparsity (Global)

Gini coefficient of the single global hybrid feature-importance vector (from Section 8). Higher values mean the global explanation is concentrated on a few dominant features rather than spread evenly across all ten.

In [ ]:
def gini_coefficient(values):
    values = np.sort(np.abs(np.asarray(values, dtype=np.float64)))
    n = len(values)
    if n == 0 or np.sum(values) == 0:
        return 0.0
    cumulative = np.cumsum(values)
    return float((n + 1 - 2 * np.sum(cumulative) / cumulative[-1]) / n)


global_sparsity = gini_coefficient(hybrid_importance)

print("=" * 60)
print("SPARSITY (GLOBAL)")
print("=" * 60)
print(f"Global Sparsity (Gini coefficient): {global_sparsity:.4f}")
print("=" * 60)

### 12. Deletion and Insertion AUC (Global Faithfulness)

Fidelity, Agreement, and Sparsity all evaluate the SHAPE of `hybrid_importance` or how well the surrogate approximates the real model's OUTPUT -- none of them directly test whether the features `hybrid_importance` ranks as important are actually the features the real PPO model relies on. The deletion/insertion test (Petsiuk et al., 2018, "RISE"; Samek et al., 2016, AOPC) answers that directly by perturbing the REAL model's input -- not the surrogate -- and watching how its own output probability responds. This is the GLOBAL analog of the same test in `lime_eval.ipynb`: instead of one feature ranking per patient, there is a SINGLE global ranking (`hybrid_importance`, from Section 8) applied identically to every row in the reference dataset.

- **Deletion**: rank the 10 features by `hybrid_importance`, most important first (same order for every patient). For each reference row, replace features one at a time with a neutral baseline (per-feature median/mode across the reference dataset), re-querying the real PPO model after each replacement, and track P(real_model_predictions) after each step. Average across ALL reference rows. A faithful global ranking should cause the average probability to collapse quickly -- **lower area under this curve is better**.
- **Insertion**: the mirror test -- start every row fully baselined and restore features one at a time in the SAME global order, tracking how quickly the average probability climbs back up. **Higher area under this curve is better**.
- Both are computed against the REAL model's probability for whatever class it actually predicted for that row (`real_model_predictions`), so this checks whether the model's own decision is faithfully explained by the global ranking -- not whether the HRFLM surrogate's own decision is.

In [ ]:
FEATURE_BASELINE_VALUES = {}
for _col in CANONICAL_FEATURE_ORDER:
    if _col in ("restecg", "slope", "sex", "cp", "exang", "fbs"):
        FEATURE_BASELINE_VALUES[_col] = float(reference_df[_col].mode().iloc[0])
    else:
        FEATURE_BASELINE_VALUES[_col] = float(reference_df[_col].median())

print("Feature baseline values (median/mode):")
for _col, _val in FEATURE_BASELINE_VALUES.items():
    print(f"  {_col:10s} {_val}")

# Single global feature ranking, descending |hybrid_importance| -- the SAME
# order applied to every reference row (unlike lime_eval.ipynb, which ranks
# features separately per patient).
global_feature_rank = list(np.argsort(-np.abs(hybrid_importance)))
print("\nGlobal feature deletion/insertion order (most -> least important):")
for _idx in global_feature_rank:
    print(f"  {CANONICAL_FEATURE_ORDER[_idx]:10s} importance={hybrid_importance[_idx]:.4f}")


def auc_of_curve(y_values):
    """Trapezoidal area under a curve sampled at len(y_values) evenly-spaced
    x-points across [0, 1] (0 = no features removed/restored, 1 = all removed/restored)."""
    if len(y_values) < 2:
        return float(y_values[0]) if y_values else 0.0
    x_values = np.linspace(0.0, 1.0, num=len(y_values))
    return float(np.trapz(y_values, x_values))


def deletion_curve_global(raw_row, target_class):
    working_row = raw_row.copy()
    probabilities = [predict_probability_from_raw(np.array([working_row]))[0][target_class]]
    for feature_index in global_feature_rank:
        feature_name = CANONICAL_FEATURE_ORDER[feature_index]
        working_row[feature_index] = FEATURE_BASELINE_VALUES[feature_name]
        probabilities.append(predict_probability_from_raw(np.array([working_row]))[0][target_class])
    return probabilities


def insertion_curve_global(raw_row, target_class):
    working_row = np.array(
        [FEATURE_BASELINE_VALUES[c] for c in CANONICAL_FEATURE_ORDER], dtype=np.float32
    )
    probabilities = [predict_probability_from_raw(np.array([working_row]))[0][target_class]]
    for feature_index in global_feature_rank:
        working_row[feature_index] = raw_row[feature_index]
        probabilities.append(predict_probability_from_raw(np.array([working_row]))[0][target_class])
    return probabilities


# Evaluated on a random sample of the reference dataset rather than all rows,
# since each row requires 11 real-model forward passes for deletion PLUS 11
# more for insertion; sampling keeps this notebook's runtime reasonable while
# still giving a stable population-level estimate.
NUM_DELETION_INSERTION_ROWS = 60
_di_rng = np.random.default_rng(RANDOM_SEED)
_di_indices = _di_rng.choice(len(X_raw), size=min(NUM_DELETION_INSERTION_ROWS, len(X_raw)), replace=False)

deletion_aucs = []
insertion_aucs = []
per_row_records = []  # one record per sampled row, before averaging

for _row_index in _di_indices:
    raw_row = X_raw[_row_index].copy()
    target_class = int(real_model_predictions[_row_index])
    original_confidence = float(real_model_probabilities[_row_index][target_class])

    row_deletion_curve = deletion_curve_global(raw_row, target_class)
    row_insertion_curve = insertion_curve_global(raw_row, target_class)
    row_deletion_auc = auc_of_curve(row_deletion_curve)
    row_insertion_auc = auc_of_curve(row_insertion_curve)

    deletion_aucs.append(row_deletion_auc)
    insertion_aucs.append(row_insertion_auc)

    per_row_records.append({
        "Row Index": int(_row_index),
        "Predicted Class": ["No Heart Disease", "Heart Disease"][target_class],
        "Original Confidence": original_confidence,
        "Deletion AUC": row_deletion_auc,
        "Insertion AUC": row_insertion_auc,
        # Full step-by-step confidence curves, kept for plotting one row's
        # collapse/recovery curve (see Section 13 below).
        "Deletion Curve": row_deletion_curve,
        "Insertion Curve": row_insertion_curve,
    })

global_deletion_auc = float(np.mean(deletion_aucs))
global_insertion_auc = float(np.mean(insertion_aucs))

print("=" * 60)
print("DELETION / INSERTION (GLOBAL FAITHFULNESS)")
print("=" * 60)
print(f"Rows evaluated                               : {len(_di_indices)}")
print(f"Global Deletion AUC (lower = more faithful)  : {global_deletion_auc:.4f}")
print(f"Global Insertion AUC (higher = more faithful) : {global_insertion_auc:.4f}")
print("=" * 60)

### 13. Plotting One Row's Deletion / Insertion Curve

`global_deletion_auc` / `global_insertion_auc` are averages over `NUM_DELETION_INSERTION_ROWS` sampled reference rows -- this cell plots ONE of those rows' actual curve so the shape behind the averaged number is visible, not just the final scalar. Since HRFLM uses a SINGLE global feature ranking (unlike LIME, where every patient gets a different ranking), the shape differences you see across rows here come entirely from each row's own feature VALUES, not from a different ranking order.

`PLOT_ROW_POSITION` is an index into `per_row_records` (0 to `NUM_DELETION_INSERTION_ROWS - 1`), NOT the row's original position in `reference_df` (that original position is shown separately as "Row Index" in the printed output).

In [ ]:
PLOT_ROW_POSITION = 0  # change to inspect a different sampled row (0 to NUM_DELETION_INSERTION_ROWS - 1)

plotted_record = per_row_records[PLOT_ROW_POSITION]
deletion_curve_to_plot = plotted_record["Deletion Curve"]
insertion_curve_to_plot = plotted_record["Insertion Curve"]
num_steps = len(deletion_curve_to_plot)
x_axis = np.linspace(0.0, 1.0, num=num_steps)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(x_axis, deletion_curve_to_plot, marker="o", color="tab:red")
axes[0].fill_between(x_axis, deletion_curve_to_plot, alpha=0.2, color="tab:red")
axes[0].set_title(f"Deletion Curve -- Reference Row {plotted_record['Row Index']}\nAUC = {plotted_record['Deletion AUC']:.4f} (lower is better)")
axes[0].set_xlabel("Fraction of features removed (global order)")
axes[0].set_ylabel(f"P({plotted_record['Predicted Class']})")
axes[0].set_ylim(0, 1)
axes[0].grid(alpha=0.3)

axes[1].plot(x_axis, insertion_curve_to_plot, marker="o", color="tab:green")
axes[1].fill_between(x_axis, insertion_curve_to_plot, alpha=0.2, color="tab:green")
axes[1].set_title(f"Insertion Curve -- Reference Row {plotted_record['Row Index']}\nAUC = {plotted_record['Insertion AUC']:.4f} (higher is better)")
axes[1].set_xlabel("Fraction of features restored (global order)")
axes[1].set_ylabel(f"P({plotted_record['Predicted Class']})")
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.3)

fig.suptitle(f"Reference Row {plotted_record['Row Index']} -- Real model predicted: {plotted_record['Predicted Class']} "
             f"(original confidence {plotted_record['Original Confidence']:.4f})")
plt.tight_layout()
plt.show()

print("=" * 60)
print(f"REFERENCE ROW {plotted_record['Row Index']} -- DELETION/INSERTION CURVE VALUES")
print("=" * 60)
print(f"Predicted class       : {plotted_record['Predicted Class']}")
print(f"Original confidence   : {plotted_record['Original Confidence']:.4f}")
print(f"Deletion AUC          : {plotted_record['Deletion AUC']:.4f}")
print(f"Insertion AUC         : {plotted_record['Insertion AUC']:.4f}")
print("-" * 60)
print(f"{'Step':>5s} {'Deletion P':>12s} {'Insertion P':>12s}")
for step_index in range(num_steps):
    print(f"{step_index:>5d} {deletion_curve_to_plot[step_index]:>12.4f} {insertion_curve_to_plot[step_index]:>12.4f}")
print("=" * 60)

### 14. Plotting the AVERAGE Deletion / Insertion Curve (All Sampled Rows)

Section 13 plots ONE row's curve, which can look very different from the population-level average if that particular row happens to be an outlier (e.g. a row whose real driving feature is ranked LAST by the global ranking, so its curve barely moves until the final step). This cell instead averages ALL `NUM_DELETION_INSERTION_ROWS` rows' curves POINT-BY-POINT (step 0 across all rows, then step 1 across all rows, etc.) into a single averaged curve, then plots THAT -- this is the curve shape that actually underlies `global_deletion_auc` / `global_insertion_auc`, rather than any one row's individual shape.

Note: the AUC of this averaged curve is close to but not always bit-for-bit identical to `global_deletion_auc`/`global_insertion_auc` (which are the AVERAGE OF EACH ROW'S OWN AUC) -- averaging curves first then taking the area, versus taking each area first then averaging, are two different (but very similar in practice) ways to summarize the same 60 curves. Both are printed below so you can compare them directly.

In [ ]:
# Stack all 60 rows' curves into a (60, 11) matrix, then average DOWN each
# column (i.e. average step 0 across all rows, average step 1 across all
# rows, ...) to get one 11-point averaged curve.
all_deletion_curves = np.array([record["Deletion Curve"] for record in per_row_records])
all_insertion_curves = np.array([record["Insertion Curve"] for record in per_row_records])

avg_deletion_curve = all_deletion_curves.mean(axis=0)
avg_insertion_curve = all_insertion_curves.mean(axis=0)

auc_of_avg_deletion_curve = auc_of_curve(avg_deletion_curve)
auc_of_avg_insertion_curve = auc_of_curve(avg_insertion_curve)

num_steps = len(avg_deletion_curve)
x_axis = np.linspace(0.0, 1.0, num=num_steps)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(x_axis, avg_deletion_curve, marker="o", color="tab:red")
axes[0].fill_between(x_axis, avg_deletion_curve, alpha=0.2, color="tab:red")
axes[0].set_title(f"AVERAGE Deletion Curve -- {len(per_row_records)} rows\nAUC of averaged curve = {auc_of_avg_deletion_curve:.4f}")
axes[0].set_xlabel("Fraction of features removed (global order)")
axes[0].set_ylabel("Mean P(each row's own predicted class)")
axes[0].set_ylim(0, 1)
axes[0].grid(alpha=0.3)

axes[1].plot(x_axis, avg_insertion_curve, marker="o", color="tab:green")
axes[1].fill_between(x_axis, avg_insertion_curve, alpha=0.2, color="tab:green")
axes[1].set_title(f"AVERAGE Insertion Curve -- {len(per_row_records)} rows\nAUC of averaged curve = {auc_of_avg_insertion_curve:.4f}")
axes[1].set_xlabel("Fraction of features restored (global order)")
axes[1].set_ylabel("Mean P(each row's own predicted class)")
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.3)

fig.suptitle("Average curve across all sampled rows -- shape behind the Section 12 summary numbers")
plt.tight_layout()
plt.show()

print("=" * 70)
print("AVERAGE DELETION / INSERTION CURVE -- VALUES AND BOTH AUC DEFINITIONS")
print("=" * 70)
print(f"Rows averaged                                        : {len(per_row_records)}")
print(f"AUC of the AVERAGED curve (this cell)                : "
      f"Deletion={auc_of_avg_deletion_curve:.4f}  Insertion={auc_of_avg_insertion_curve:.4f}")
print(f"AVERAGE of each row's own AUC (Section 12, for comparison) : "
      f"Deletion={global_deletion_auc:.4f}  Insertion={global_insertion_auc:.4f}")
print("-" * 70)
print(f"{'Step':>5s} {'Avg Deletion P':>15s} {'Avg Insertion P':>16s}")
for step_index in range(num_steps):
    print(f"{step_index:>5d} {avg_deletion_curve[step_index]:>15.4f} {avg_insertion_curve[step_index]:>16.4f}")
print("=" * 70)

### 15. Reward Improvement (Global)

Uses the SAME reward rule the PPO agent was trained with (`HeartDiseaseEnvironment.step()` in `PPO.ipynb`: reward = +1 if the chosen action equals the true label, else -1), computed across ALL reference rows for both the real PPO policy and the HRFLM surrogate's decisions.

In [ ]:
def step_reward(action, true_label):
    """Matches HeartDiseaseEnvironment.step() in PPO.ipynb exactly."""
    return 1 if action == true_label else -1


ppo_rewards = [step_reward(pred, true) for pred, true in zip(real_model_predictions, y_true)]
surrogate_rewards = [step_reward(pred, true) for pred, true in zip(hybrid_predictions, y_true)]

mean_ppo_reward = float(np.mean(ppo_rewards))
mean_surrogate_reward = float(np.mean(surrogate_rewards))
global_reward_improvement = mean_ppo_reward - mean_surrogate_reward

print("=" * 60)
print("REWARD IMPROVEMENT (GLOBAL)")
print("=" * 60)
print(f"Mean Reward -- Real PPO Policy         : {mean_ppo_reward:+.4f}")
print(f"Mean Reward -- HRFLM Global Surrogate  : {mean_surrogate_reward:+.4f}")
print(f"Reward Improvement (PPO - Surrogate)   : {global_reward_improvement:+.4f}")
print("=" * 60)

### 16. Summary and Export

Saved alongside `models/hrflm/<version>/hrflm_report.json` so the admin-only backend view can also surface these explainability-quality metrics, not just the feature-importance ranking.

In [ ]:
hrflm_metrics_summary = pd.DataFrame([{
    "Fidelity": global_fidelity,
    "Accuracy Gain": global_accuracy_gain,
    "Agreement": global_agreement,
    "Stability": global_stability,
    "Sparsity": global_sparsity,
    "Deletion AUC": global_deletion_auc,
    "Insertion AUC": global_insertion_auc,
    "Reward Improvement": global_reward_improvement,
}])

print("=" * 70)
print("HRFLM GLOBAL EXPLAINABILITY METRICS SUMMARY")
print("=" * 70)
print(hrflm_metrics_summary.to_string(index=False))
print("=" * 70)

hrflm_metrics_summary.to_csv("hrflm_eval_explainability_metrics.csv", index=False)
print("\nSaved: hrflm_eval_explainability_metrics.csv")

# Also save alongside the exported HRFLM artifact directory, if one exists,
# so the admin-only backend view (hrflmService.js) can serve these metrics
# together with the feature-importance report.
import glob
hrflm_dirs = sorted(glob.glob(os.path.join("..", "..", "models", "hrflm", "hrflm-*")))
if hrflm_dirs:
    target_dir = hrflm_dirs[-1]
    metrics_path = os.path.join(target_dir, "hrflm_eval_metrics.json")
    with open(metrics_path, "w") as f:
        json.dump(hrflm_metrics_summary.iloc[0].to_dict(), f, indent=2)
    print(f"Also saved: {metrics_path}")
else:
    print("No models/hrflm/hrflm-* directory found -- run HRFLM.ipynb first to export one.")